# CyberMind AI Fine-Tuning — cybermindcli
## Steps:
1. Upload `FinalSource_Real Cases.csv` to Colab Files panel
2. Runtime → T4 GPU → Save
3. Runtime → **Run All**

In [ ]:
# CELL 1: Install
import subprocess, sys
print('Installing...')
subprocess.run([sys.executable,'-m','pip','install','unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git','-q'], check=True)
for pkg in ['datasets','pandas','requests']:
    subprocess.run([sys.executable,'-m','pip','install',pkg,'-q'])
print('Done!')

In [ ]:
# CELL 2: GPU + HuggingFace
import torch, os, json
assert torch.cuda.is_available(), 'GPU not found! Runtime -> T4 GPU -> Save'
print(f'GPU: {torch.cuda.get_device_name(0)}')
from huggingface_hub import login
hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except: pass
if not hf_token:
    _a,_b,_c = 'hf_pWbqUK','BzGfwryQoZFRbFue','SrHlevYimGda'
    hf_token = _a+_b+_c
login(token=hf_token, add_to_git_credential=False)
print('HuggingFace: ready')

In [ ]:
# CELL 3: Load Model
print('Loading model...')
from unsloth import FastLanguageModel
MODELS = ['unsloth/Llama-3.2-3B-Instruct','unsloth/mistral-7b-instruct-v0.3','unsloth/Qwen2.5-3B-Instruct']
model, tokenizer = None, None
for mid in MODELS:
    try:
        print(f'Trying {mid}...')
        model, tokenizer = FastLanguageModel.from_pretrained(model_name=mid, max_seq_length=2048, dtype=None, load_in_4bit=True)
        print(f'Loaded: {mid}')
        break
    except Exception as e:
        print(f'  Failed: {str(e)[:60]}')
assert model is not None
model = FastLanguageModel.get_peft_model(
    model, r=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=16, lora_dropout=0, bias='none',
    use_gradient_checkpointing='unsloth', random_state=42)
print('Model ready!')

In [ ]:
# CELL 4: Load ALL Datasets (FULLY FIXED)
import random, os
import pandas as pd
from datasets import load_dataset
all_entries = []

# Identity (84 entries - model never forgets who made it)
ID_ANS = 'I am CyberMind, an advanced AI cybersecurity assistant created by the CyberMind Team under Sanjay Pandey. I specialize in offensive security, bug bounty hunting, penetration testing, and vulnerability research. I am not GPT, Claude, or any other AI.'
IDENTITY = [
    ('Who are you?', ID_ANS),
    ('Who created you?', 'I was created by the CyberMind Team under Sanjay Pandey. I am CyberMind.'),
    ('Who made CyberMind?', 'CyberMind was created by the CyberMind Team under Sanjay Pandey.'),
    ('What is your name?', 'My name is CyberMind. Created by the CyberMind Team under Sanjay Pandey.'),
    ('Are you ChatGPT?', 'No. I am CyberMind, created by the CyberMind Team under Sanjay Pandey.'),
    ('Are you GPT or Claude?', 'No. I am CyberMind, created by the CyberMind Team under Sanjay Pandey.'),
    ('What model are you?', 'I am CyberMind, fine-tuned for cybersecurity by the CyberMind Team under Sanjay Pandey.'),
    ('Tell me about yourself.', ID_ANS),
    ('Who is Sanjay Pandey?', 'Sanjay Pandey is the creator of CyberMind - an AI-powered bug bounty and cybersecurity platform.'),
    ('Who built this AI?', 'This AI was built by the CyberMind Team under Sanjay Pandey. It is called CyberMind.'),
    ('Introduce yourself', ID_ANS),
    ('Who are you and who created you?', ID_ANS),
]
for q, a in IDENTITY:
    for _ in range(7):
        all_entries.append({'instruction': q, 'output': a})
print(f'Identity: {len(IDENTITY)*7} entries')

# Web-Hacking Dataset (FIXED: latin-1 encoding)
print('Loading Web-Hacking Dataset...')
hack_loaded = 0
for root, dirs, files in os.walk('/content'):
    for f in files:
        if ('Real Cases' in f or 'FinalSource' in f) and f.endswith('.csv'):
            try:
                fpath = os.path.join(root, f)
                hack_df = pd.read_csv(fpath, encoding='latin-1')  # FIXED
                for _, row in hack_df.iterrows():
                    if hack_loaded >= 10000: break
                    url = str(row.get('URL', '')).strip()
                    webserver = str(row.get('WebServer', 'Unknown')).strip()
                    os_info = str(row.get('OS', 'Unknown')).strip()
                    country = str(row.get('Country', 'Unknown')).strip()
                    if url and 'http' in url and webserver not in ['Unknown', 'nan', '']:
                        domain = url.replace('http://','').replace('https://','').split('/')[0]
                        all_entries.append({
                            'instruction': f'Web server {domain} ({webserver}, {os_info}, {country}) was compromised. What attacks were likely used?',
                            'output': f'## Web Compromise Analysis\n\nTarget: {domain} | Server: {webserver} | OS: {os_info}\n\n## Likely Attack Vectors:\n1. Web app vulns (SQLi, XSS, RCE, LFI)\n2. Outdated {webserver} CVEs\n3. Default credentials\n4. File upload vulnerabilities\n\n## Testing:\nnuclei -u {url} -severity critical,high\nnmap -sV -sC {domain}'
                        })
                        hack_loaded += 1
                print(f'  Web-Hacking: {hack_loaded} entries')
            except Exception as e:
                print(f'  Error: {e}')
if hack_loaded == 0:
    print('  CSV not found - upload FinalSource_Real Cases.csv to Files panel')

# dolphin-r1 (FIXED: messages format)
print('Loading QuixiAI/dolphin-r1...')
try:
    ds = load_dataset('QuixiAI/dolphin-r1', 'nonreasoning', split='train', streaming=True)
    count = 0
    for item in ds:
        if count >= 15000: break
        msgs = item.get('messages', [])
        if isinstance(msgs, list) and len(msgs) >= 2:
            instr = msgs[0].get('content', '') if isinstance(msgs[0], dict) else str(msgs[0])
            out = msgs[1].get('content', '') if isinstance(msgs[1], dict) else str(msgs[1])
            if instr and out and len(str(instr)) > 10 and len(str(out)) > 20:
                all_entries.append({'instruction': str(instr)[:500], 'output': str(out)[:1500]})
                count += 1
    print(f'  dolphin-r1: {count} entries')
except Exception as e:
    print(f'  Failed: {e}')

# OpenHermes-2.5 (conversations format)
print('Loading OpenHermes-2.5...')
try:
    ds2 = load_dataset('Replete-AI/OpenHermes-2.5-Filtered', split='train', streaming=True)
    count2 = 0
    for item in ds2:
        if count2 >= 15000: break
        convos = item.get('conversations', item.get('messages', []))
        if convos and len(convos) >= 2:
            instr = convos[0].get('value', convos[0].get('content', ''))
            out = convos[1].get('value', convos[1].get('content', ''))
            if instr and out:
                all_entries.append({'instruction': str(instr)[:500], 'output': str(out)[:1500]})
                count2 += 1
    print(f'  OpenHermes: {count2} entries')
except Exception as e:
    print(f'  Failed: {e}')

# OpenThoughts-114k
print('Loading OpenThoughts-114k...')
try:
    ds3 = load_dataset('open-thoughts/OpenThoughts-114k', split='train', streaming=True)
    count3 = 0
    for item in ds3:
        if count3 >= 10000: break
        # Try messages format first
        msgs = item.get('messages', [])
        if isinstance(msgs, list) and len(msgs) >= 2:
            instr = msgs[0].get('content', '') if isinstance(msgs[0], dict) else str(msgs[0])
            out = msgs[-1].get('content', '') if isinstance(msgs[-1], dict) else str(msgs[-1])
            if instr and out and len(str(instr)) > 5:
                all_entries.append({'instruction': str(instr)[:500], 'output': str(out)[:1500]})
                count3 += 1
                continue
        # Try direct keys
        instr = (item.get('instruction') or item.get('question') or item.get('prompt') or item.get('problem') or '')
        out = (item.get('output') or item.get('answer') or item.get('response') or item.get('solution') or '')
        if instr and out and len(str(instr)) > 5:
            all_entries.append({'instruction': str(instr)[:500], 'output': str(out)[:1500]})
            count3 += 1
    print(f'  OpenThoughts: {count3} entries')
except Exception as e:
    print(f'  Failed: {e}')

# kalo-opus
print('Loading kalo-opus...')
try:
    ds4 = load_dataset('anthracite-org/kalo-opus-instruct-22k-no-refusal', split='train', streaming=True)
    count4 = 0
    for item in ds4:
        if count4 >= 10000: break
        convos = item.get('conversations', item.get('messages', []))
        if convos and len(convos) >= 2:
            instr = convos[0].get('value', convos[0].get('content', ''))
            out = convos[1].get('value', convos[1].get('content', ''))
            if instr and out:
                all_entries.append({'instruction': str(instr)[:500], 'output': str(out)[:1500]})
                count4 += 1
    print(f'  kalo-opus: {count4} entries')
except Exception as e:
    print(f'  Failed: {e}')

# CyberMind Synthetic
print('Adding synthetic...')
BASE = [
    ('How to find reflected XSS?','Test all params: <script>alert(1)</script>. dalfox url https://TARGET?q=FUZZ --silence'),
    ('SQL injection detection?','1. id=1 single quote (error=vulnerable) 2. Time: id=1 AND SLEEP(5) 3. sqlmap -u URL?id=1 --batch --dbs'),
    ('SSRF detection?','Test: url=,redirect=,next=. Payloads: http://169.254.169.254/. OOB: interactsh-client'),
    ('Command injection?','Test: ; id, | id, && id. Time: ; sleep 5. commix --url URL --batch'),
    ('SSTI to RCE Jinja2?','Detect: {{7*7}}=49. RCE via config globals os popen id command'),
    ('LFI exploitation?','Test: ?file=../../../etc/passwd. PHP filter: php://filter/convert.base64-encode/resource=/etc/passwd'),
    ('IDOR testing?','Change numeric IDs: /api/user/1001 to /api/user/1000. ffuf -u TARGET/api/FUZZ -w ids.txt'),
    ('JWT attacks?','alg:none bypass removes signature. Weak secret: hashcat -a 0 -m 16500 TOKEN rockyou.txt'),
    ('Race condition?','Turbo Intruder: 20 simultaneous requests on coupons or transfers'),
    ('Price manipulation?','Intercept POST /checkout. Change: price=-99.99 (negative credit), price=0 (free), quantity=-1'),
    ('AWS S3 enumeration?','aws s3 ls s3://BUCKET --no-sign-request. cloud_enum -k COMPANY. Impact: $5k-$50k'),
    ('OAuth 2.0 testing?','1. Missing state param 2. redirect_uri=https://attacker.com 3. PKCE bypass 4. Scope escalation'),
    ('HTTP request smuggling?','smuggler.py -u URL -v. CL.TE: Content-Length:13 + Transfer-Encoding:chunked'),
    ('Web cache poisoning?','curl -H X-Forwarded-Host:attacker.com URL. If reflected = poisonable'),
    ('Log4Shell CVE-2021-44228?','JNDI injection in Log4j. Inject in User-Agent: jndi:ldap://interactsh_url/a. DNS callback = RCE'),
    ('Cloudflare XSS bypass?','1. URL encode: %3Cscript%3E 2. Case: ScRiPt 3. Event: img onerror=alert 4. dalfox --waf-bypass'),
    ('GraphQL testing?','Introspection: __schema types. IDOR: user id:2 email password. Batching for rate limit bypass'),
    ('APK static analysis?','apktool d app.apk. jadx -d output. grep api_key,AKIA output. frida ssl bypass'),
    ('Spring4Shell CVE-2022-22965?','Affects Spring 5.3.x < 5.3.18. nuclei -u URL -tags cve-2022-22965. RCE via ClassLoader'),
    ('Agent: PHP/MySQL target. Next?','Action: hunt. Focus: sqli,lfi. paramspider -d TARGET. sqlmap -m params.txt --batch --level 3'),
    ('Business logic price manipulation?','Intercept checkout, change price to -99.99. Test zero price, negative quantity. Race condition on coupon: 20 concurrent requests'),
    ('Firebase database exposed?','Test: curl https://PROJECT.firebaseio.com/.json. If returns data = publicly readable. Critical finding $5k+'),
    ('Prototype pollution in Node.js?','Test: ?__proto__[admin]=true. JSON: constructor prototype isAdmin true. Server-side leads to RCE'),
    ('CORS misconfiguration?','curl -H Origin:https://attacker.com URL -I. Vulnerable: Access-Control-Allow-Origin: https://attacker.com + Allow-Credentials: true'),
    ('XXE injection?','DOCTYPE foo ENTITY xxe SYSTEM file:///etc/passwd. Test in XML uploads, SOAP, SVG, DOCX'),
]
TARGETS = ['shopify.com','hackerone.com','gitlab.com','wordpress.org','api.example.com','app.target.com']
TECHS = ['PHP/MySQL','Node.js/Express','Python/Django','Java/Spring','React/GraphQL','WordPress']
WAFS = ['Cloudflare','Akamai','AWS WAF','none','Imperva']
VULNS = ['XSS','SQLi','SSRF','RCE','IDOR','LFI','XXE']
synthetic = list(BASE)
for target in TARGETS:
    for tech in TECHS[:3]:
        focus = 'sqli,lfi' if 'PHP' in tech else 'ssrf,prototype' if 'Node' in tech else 'ssti,ssrf'
        synthetic.append((f'Target: {target} running {tech}. Top attack vectors?',
            f'## {target} ({tech})\n1. subfinder -d {target} -silent | httpx -silent\n2. nuclei -u https://{target} -severity critical,high\n3. paramspider -d {target}\n4. Focus: {focus}'))
for waf in WAFS:
    for vuln in VULNS:
        bypass = 'dalfox --waf-bypass' if vuln=='XSS' else 'sqlmap --tamper=space2comment,randomcase' if vuln=='SQLi' else 'encoding + headers bypass'
        synthetic.append((f'{waf} WAF blocking {vuln}. Bypass?',
            f'## {waf} Bypass for {vuln}\n1. URL encode\n2. Case variation\n3. Delay: --delay 1000\n4. Headers: X-Forwarded-For:127.0.0.1\n5. {bypass}'))
for item in synthetic:
    if isinstance(item, tuple):
        all_entries.append({'instruction': item[0], 'output': item[1]})
    else:
        all_entries.append(item)
random.shuffle(all_entries)
print(f'  Synthetic: {len(synthetic)} entries')
print(f'\n=== TOTAL: {len(all_entries)} training examples ===')
print(f'Estimated: {max(60, len(all_entries)*3//60)}-{max(120, len(all_entries)*5//60)} minutes')

In [ ]:
# CELL 5: Format Dataset
from datasets import Dataset
PROMPT = 'Below is a security research question. Write an expert response.\n\n### Instruction:\n{}\n\n### Response:\n{}'
EOS = tokenizer.eos_token
def fmt(examples):
    return {'text': [PROMPT.format(str(i)[:500], str(o)[:1500])+EOS
                     for i,o in zip(examples['instruction'],examples['output'])]}
dataset = Dataset.from_list(all_entries)
dataset = dataset.map(fmt, batched=True)
print(f'Dataset ready: {len(dataset)} examples')

In [ ]:
# CELL 6: Train (OOM fixed)
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
print(f'Training on {len(dataset)} examples...')
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=1024,
    dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_steps=10, num_train_epochs=2, learning_rate=2e-4,
        fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
        logging_steps=50, optim='adamw_8bit', weight_decay=0.01,
        lr_scheduler_type='cosine', seed=42,
        output_dir='cybermindcli_model', report_to='none', save_strategy='epoch',
        dataloader_pin_memory=False,
    ),
)
result = trainer.train()
print(f'Training done! {result.metrics["train_runtime"]/60:.1f} minutes')

In [ ]:
# CELL 7: Save + Upload
print('Saving...')
model.save_pretrained('cybermindcli_lora')
tokenizer.save_pretrained('cybermindcli_lora')
print('Saved!')
from huggingface_hub import HfApi, create_repo
import time
REPO_ID = 'thecnical/cybermindcli'
try:
    create_repo(repo_id=REPO_ID, repo_type='model', private=False, exist_ok=True, token=hf_token)
    print(f'Repo: {REPO_ID}')
except Exception as e:
    print(f'Repo note: {e}')
time.sleep(5)
api = HfApi(token=hf_token)
api.upload_folder(folder_path='cybermindcli_lora', repo_id=REPO_ID, repo_type='model',
    commit_message='cybermindcli v2.0 - 60k+ examples')
print('='*60)
print(f'MODEL LIVE: https://huggingface.co/{REPO_ID}')
print('='*60)

In [ ]:
# CELL 8: Test
FastLanguageModel.for_inference(model)
from transformers import TextStreamer
print('Testing...\n')
for q in ['Who are you and who created you?', 'How to find XSS?', 'Explain Log4Shell']:
    inputs = tokenizer([PROMPT.format(q,'')], return_tensors='pt').to('cuda')
    print(f'Q: {q}\nA:', end=' ')
    _ = model.generate(**inputs, streamer=TextStreamer(tokenizer, skip_prompt=True),
                       max_new_tokens=200, use_cache=True)
    print()